In [4]:
# %pip install arize-phoenix openai gcsfs nest_asyncio llama-index-llms-openai "httpx<0.28"

In [4]:
import os 
from getpass import getpass 
from gcsfs import GCSFileSystem
from llama_index.core import (Settings,StorageContext, load_index_from_storage)
from llama_index.core.graph_stores import SimpleGraphStore #helps to create graph
from llama_index.embeddings.huggingface import HuggingFaceEmbedding #embeddubgs
from openinference.instrumentation import suppress_tracing  #temporarily disables tracing (OpenTelemetry)
from llama_index.llms.groq import Groq

from openinference.instrumentation.llama_index import LlamaIndexInstrumentor #tracer for LlamaIndex
from phoenix.otel import register 
from tqdm import tqdm
from dotenv import load_dotenv

/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


In [5]:
load_dotenv()

True

In [6]:
# # # %pip install llama-index-embeddings-huggingface
# # %pip install llama-index-llms-groq
# %pip install openinference-instrumentation-llama-index

In [7]:
tracer_provider=register(project_name='llm-rag-app')
LlamaIndexInstrumentor().instrument(skip_dep_check=True, tracer_provider=tracer_provider)

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: llm-rag-app
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [8]:
file_system=GCSFileSystem(project="public-assets-275721")

In [34]:
fs.ls("gcp-public-data-landsat")

['gcp-public-data-landsat/LC08',
 'gcp-public-data-landsat/LE07',
 'gcp-public-data-landsat/LM01',
 'gcp-public-data-landsat/LM02',
 'gcp-public-data-landsat/LM03',
 'gcp-public-data-landsat/LM04',
 'gcp-public-data-landsat/LM05',
 'gcp-public-data-landsat/LO08',
 'gcp-public-data-landsat/LT04',
 'gcp-public-data-landsat/LT05',
 'gcp-public-data-landsat/LT08',
 'gcp-public-data-landsat/index.csv.gz']

In [9]:
fs=GCSFileSystem(token='anon')
with fs.open("gcp-public-data-landsat/index.csv.gz",'rb') as f:
    data=f.read()

print(data[:500])

b'\x1f\x8b\x08\x08L\xde\xe8a\x02\xffout.tar\x00\xac\xbd[\x8f\xa49\x96\x1c\xf8\xde\xbf\xc5\xc3\x9d\xf7K\xbeeWW\xd7\xd4L\xdd\xb62z\xa7\xa5\x97\xc4\xac\x04\x08\x02\x04A\x80\xb4\xff\x7f\xcd\xec\x90\xdf\xc5=\xdd\x83\t\xec\xf4tW\x14\x99\x11\x19\xa4\x93\x87\xe7b\xc7\xec\xcb\x0f?\xfe\xf6\xe3\xd7\x9f\xffv\xf9\xe3\xcf\xdf\xff\xf6\x8f\x1f\xde\xf9\xe5\x97?>\xff\xf0\xe3\x0f\x7f~\xfe\xbb\xfd\xdb\x8f\xbf}\xf9\xfdO~\xf5\xb7\xcf\xef?~\xfd\xfc\xc3\xff\xf5\x8f\x9f\xff\xfc\xf1o\x97\x1f~\xff\xe5\x97\x1f\x7fx\xff\xf9\xf7\xdf\xbe\xfe\xf6\x8f_\xff\xfa\xe3\x9f\xc7\x91\x1f\xf0G\x7f\xfa\xfd\xcf\xff\xa4\xef\xfe\xf9\xb7\x9f\xbe\xbe\xff\xfc\xeb\x8f\xfc\x01\x9f\xbf\xbe\xff\xa7?~\xbc\xfc\xfb\x9f_\xbe\xfe\xf1\xf9\xfd_\xf4\xc5\x9f\xbf\xff\xfb\xe5\x87_~\xff\xc7\xdf\xbe\xfe\xf0\xfb\xff\x8d\x9f\xf3\xdb\xef\x7f\xbe\xff\xcb\xd7_>\xbf_\xbe\xfc\xfe\x8f\xf1\xd5\xbf\xff\xf8\xe5\xfd\xeb/\xbf\xffv\xf9\xf1\xf3\xf8\xe2\xfd\xf7\xf7\xcf\xbf|\xfd\xf2\xf3\x7f\xfe\xf1\xf2\xd7\xcf_~\xfc\xfa\x8f?\x7f\xf9\xcb/?V\xdf\x9c\xcb=8\xef}L\x9f\xbf

In [ ]:
index_path="arize-phoenix-assets/datasets/unstructured/llm/llama-index/arize-docs/index/"
storage_context=StorageContext.from_defaults( #Create storage context.  "Load index from THIS filesystem + THIS location"
    fs=file_system,
    persist_dir=index_path,
    graph_store=SimpleGraphStore(),#prevents unauthorixed request to gcs
)


Settings.llm=Groq(model='llama-3.1-8b-instant',
    api_key=os.environ['GROQ_API_KEY']
)

In [ ]:
#%pip install -U "transformers<4.40" "sentence-transformers<3.0"

In [11]:
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-base-en-v1.5"
)

/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [12]:
index=load_index_from_storage(storage_context)

In [16]:
query_engin=index.as_query_engine()
# Asking the Application questions about the Arize product
queries = [
    "How can I query for a monitor's status using GraphQL?",
    "How do I delete a model?",
    "How much does an enterprise license of Arize cost?",
    "How do I log a prediction using the python SDK?",
]


for q in tqdm(queries):
    rsp=query_engin.query(q)
    print(f'Query :{q}')
    print(f'response :{rsp}')


  0%|          | 0/4 [00:00<?, ?it/s]


ValueError: shapes (768,) and (1536,) not aligned: 768 (dim 0) != 1536 (dim 0)